In [ ]:
import os
import re
import gzip
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

drive.mount('/content/drive')

base_dir = "/content/drive/MyDrive/Social Network Analysis/Data"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Nạp và phân tích thông tin đồ thị

In [ ]:
graph_path = os.path.join(base_dir, "raw/amazon0302.txt")

# Nạp đồ thị dưới dạng có hướng (DiGraph) để tính chính xác In/Out Degree
print("Đang nạp đồ thị...")
G_directed = nx.read_edgelist(graph_path, nodetype=int, comments='#', create_using=nx.DiGraph())

print("Đang tính toán các chỉ số Bậc (Degree)...")
nodes_list = list(G_directed.nodes())

# Tạo DataFrame cấu trúc mạng
df_graph = pd.DataFrame({
    'product_id': nodes_list,
    'in_degree': [G_directed.in_degree(node) for node in nodes_list],
    'out_degree': [G_directed.out_degree(node) for node in nodes_list],
    'total_degree': [G_directed.degree(node) for node in nodes_list]
})

print(f"Đã tạo xong df_graph: {df_graph.shape[0]:,} dòng.")
display(df_graph.head())

Đang nạp đồ thị...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Social Network Analysis/Data/Amazon0302.txt'

# Trích xuất metadata sản phẩm

In [ ]:
print("Trích xuất và tiền xử lý dữ liệu từ file amazon-meta ---")
meta_path = os.path.join(base_dir, "raw/amazon-meta.txt")

meta_data = []

with open(meta_path, 'r', encoding='utf-8', errors='ignore') as f:
    current_item = {}
    for line in f:
        line = line.strip()

        if line.startswith("Id:"):
            if current_item and 'product_id' in current_item:
                meta_data.append(current_item)
            current_item = {'product_id': int(line.split()[1])}

        elif line.startswith("group:"):
            current_item['group'] = line.split(":", 1)[1].strip()

        elif line.startswith("salesrank:"):
            current_item['salesrank'] = int(line.split()[1])

        elif line.startswith("similar:"):
            # Chỉ lấy số lượng phần tử similar đầu tiên (số đầu tiên sau chữ similar:)
            parts = line.split()
            current_item['similar'] = int(parts[1]) if len(parts) > 1 else 0

        elif line.startswith("categories:"):
            # Chỉ lấy số lượng category (số lượng danh sách chứa sản phẩm)
            current_item['category_type_count'] = int(line.split()[1])

        elif line.startswith("reviews:"):
            # Trích xuất total reviews, downloaded, và avg rating bằng Regex
            match = re.search(r"total:\s*(\d+)\s+downloaded:\s*(\d+)\s+avg rating:\s*([\d.]+)", line)
            if match:
                current_item['total_reviews'] = int(match.group(1))
                current_item['downloaded'] = int(match.group(2))
                current_item['avg_rating'] = float(match.group(3))

    # Thêm item cuối cùng vào danh sách
    if current_item and 'product_id' in current_item:
        meta_data.append(current_item)

# Chuyển thành DataFrame
df_meta = pd.DataFrame(meta_data)

# Điền các giá trị thiếu bằng 0 hoặc chuỗi trống cho an toàn dữ liệu
df_meta['group'] = df_meta['group'].fillna('Unknown')
df_meta['salesrank'] = df_meta['salesrank'].fillna(-1)
df_meta = df_meta.fillna(0)

print(f"Đã tạo xong df_meta: {df_meta.shape[0]:,} dòng.")
display(df_meta.head())

# Đọc dữ liệu Hub Nodes

In [ ]:
import pandas as pd
import os

# Paths to the selected files
hub_csv_path = '/content/drive/MyDrive/Social Network Analysis/Data/graph_information/real_hub.csv'
superhub_csv_path = '/content/drive/MyDrive/Social Network Analysis/Data/graph_information/superhub.csv'

# Load the datasets
df_real_hub = pd.read_csv(hub_csv_path)
df_superhub_ext = pd.read_csv(superhub_csv_path)

# Display the first few rows of each
print("Real Hub Data:")
display(df_real_hub.head())
print("\nSuperhub Data:")
display(df_superhub_ext.head())

# Chuẩn hóa dữ liệu Hub

In [ ]:
# Rename 'Node' column to 'product_id' for consistency
df_real_hub = df_real_hub.rename(columns={'Node': 'product_id'})
df_superhub_ext = df_superhub_ext.rename(columns={'Node': 'product_id'})

# Verify the change
print("Columns in df_real_hub:", df_real_hub.columns.tolist())
print("Columns in df_superhub_ext:", df_superhub_ext.columns.tolist())

# Tiền xử lý danh sách Hub

In [ ]:
# Xóa cột HubScore và thêm cột node_type
df_real_hub = df_real_hub.drop(columns=['HubScore'])
df_real_hub['node_type'] = 'Hub'

df_superhub_ext = df_superhub_ext.drop(columns=['HubScore'])
df_superhub_ext['node_type'] = 'Super Hub'

# Kiểm tra kết quả
print("df_real_hub info:")
display(df_real_hub.head())
print("\ndf_superhub_ext info:")
display(df_superhub_ext.head())

# Kết hợp các nhóm Hub

In [ ]:
# 1. Kết hợp hai danh sách hub
df_hubs_combined = pd.concat([df_real_hub, df_superhub_ext], ignore_index=True)

# 2. Thực hiện join các bảng dựa trên product_id
# Sử dụng left join với df_graph làm gốc để giữ lại toàn bộ cấu trúc mạng
df_final = df_graph.merge(df_meta, on='product_id', how='left') \
                   .merge(df_hubs_combined, on='product_id', how='left')

# 3. Gán nhãn 'Normal' cho những node không nằm trong danh sách Hubs
df_final['node_type'] = df_final['node_type'].fillna('Normal')

# Hiển thị kết quả
print(f"Kích thước bảng sau khi join: {df_final.shape}")
print("Thống kê số lượng theo loại node:")
print(df_final['node_type'].value_counts())
display(df_final.head())

# Lưu kết quả phân tích

In [ ]:
# Định nghĩa đường dẫn lưu file
output_path = '/content/drive/MyDrive/Social Network Analysis/Data/graph_information/final_merged_data.csv'

# Xuất DataFrame df_final ra file CSV
df_final.to_csv(output_path, index=False)

print(f"Đã xuất dữ liệu thành công tại: {output_path}")